# CUDA Graph Execution Replay in Low-Latency LLM Serving

## 1. The CPU Launch Bottleneck in Autoregressive Decoding
During the autoregressive decoding phase of an LLM, generating a single token requires executing 50 to 100 small GPU kernels (LayerNorm, QKV projection, Attention, Softmax, FFN, Residual add) for each layer.

On high-end GPUs (e.g. NVIDIA H100):
- Actual GPU execution time per kernel: $\approx 2-5 \, \mu s$.
- CPU driver overhead to enqueue a kernel via CUDA API: $\approx 8-15 \, \mu s$.

Because **CPU launch overhead exceeds GPU execution time**, the GPU hardware warps sit idle waiting for the next CPU instruction! The serving engine becomes severely **CPU-bound** during token-by-token decoding.

### CUDA Graphs Solution
CUDA Graphs captures a complete sequence of GPU kernel launches, parameter bindings, and dependency edges into a single static **Execution Graph (DAG)**:
1. **Capture Phase**: Record the forward pass once into a graph object.
2. **Instantiate Phase**: Compile the graph into an executable hardware graph.
3. **Replay Phase**: The CPU launches the **entire multi-layer forward pass with a single PCIe register write**!

$$\text{CPU Overhead Reduction: } \approx 1.2\text{ms} \longrightarrow < 10\mu s$$

In [ ]:
import numpy as np

class SimulatedCUDAGraphEngine:
    """
    Demonstrates CUDA Graph record and replay mechanics.
    Shows zero-allocation static memory buffer reuse and launch overhead elimination.
    """
    def __init__(self, hidden_dim=64):
        self.hidden_dim = hidden_dim
        # Static persistent memory buffers
        self.static_input = np.zeros(hidden_dim, dtype=np.float32)
        self.static_weight = np.random.randn(hidden_dim, hidden_dim).astype(np.float32) * 0.1
        self.static_output = np.zeros(hidden_dim, dtype=np.float32)
        
        self.graph_recorded = False

    def capture_graph(self):
        """Simulates cudaStreamBeginCapture() and cudaStreamEndCapture()."""
        print("[CUDA Graph] Capturing kernel operations into static DAG...")
        # Static graph execution logic
        self.graph_op = lambda: np.dot(self.static_input, self.static_weight, out=self.static_output)
        self.graph_recorded = True
        print("[CUDA Graph] Graph instantiated and compiled successfully!")

    def replay(self, dynamic_input_vector):
        """Simulates cudaGraphLaunch() with static buffer copying."""
        assert self.graph_recorded, "Must capture graph prior to replay!"
        
        # 1. Fast copy dynamic data into pre-pinned static input buffer (cudaMemcpyAsync)
        np.copyto(self.static_input, dynamic_input_vector)
        
        # 2. Launch whole graph with single hardware call
        self.graph_op()
        
        return self.static_output

# Demonstration
engine = SimulatedCUDAGraphEngine(hidden_dim=32)
engine.capture_graph()

# Replay forward pass across multiple steps
x_sample = np.ones(32) * 2.5
out = engine.replay(x_sample)

print(f"Input Vector Norm  : {np.linalg.norm(x_sample):.2f}")
print(f"Replay Output Norm : {np.linalg.norm(out):.2f}")
print("CUDA Graph replay executed with zero memory allocations and minimal CPU overhead!")

## 2. Managing Variable Batch Sizes via Bucket Graphs
CUDA Graphs require **strictly static tensor shapes and memory addresses**; dynamic tensor resizing invalidates the graph.

Serving engines (vLLM, TensorRT-LLM) resolve this using **Batch Bucketing**:
- Pre-capture discrete graph templates for batch sizes: $\{1, 2, 4, 8, 16, 32\}$.
- If current batch size is 7, the engine selects the bucket for **batch size 8**, pads 1 dummy sequence, and executes the compiled CUDA Graph with zero launch latency.

## 3. High-Yield Interview Questions & Best Practices

### Q1: Why can't CUDA Graphs be used directly during the prefill phase?
**Answer:** In the prefill phase, sequence lengths vary widely (e.g. 150 tokens vs 8200 tokens). Compiling a CUDA Graph for every possible sequence length would exhaust GPU memory with graph instantiation metadata. Furthermore, prefill is compute-bound (dominated by large GEMM matrix multiplications where GPU compute time far exceeds CPU launch overhead), so the benefit of CUDA Graph launch reduction is negligible.

### Q2: What operations cause CUDA Graph capture to fail?
**Answer:**
1. **Host-Device Synchronization**: Any call to `cudaStreamSynchronize()`, `cudaMemcpy` from device to host, or Python branching based on GPU tensor values (`if tensor.item() > 0:`).
2. **Dynamic Memory Allocation**: Invoking `cudaMalloc` or `cudaFree` inside the captured stream. All memory buffers must be pre-allocated statically.